In [ ]:
import torch
import matplotlib.pyplot as plt
from config import Config
from datasets.celeba_dataset import CelebAInpaintingDataset
from models.cae import CAE

cfg = Config()

device = (
    torch.device('cuda') if torch.cuda.is_available()
    else torch.device('mps') if torch.backends.mps.is_available()
    else torch.device('cpu')
)
print('device:', device)


In [ ]:
dataset = CelebAInpaintingDataset(root='./data', split='test', image_size=cfg.image_size)

def denorm(t):
    return (t * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).cpu().numpy()


In [ ]:
# 모델 로드 — 체크포인트 경로를 맞게 수정하세요
models = {
    'CAE': CAE(),
    # 'PConv': PConvCAE(),   # 나중에 추가
    # 'Gated': GatedCAE(),   # 나중에 추가
}

checkpoints = {
    'CAE': 'checkpoints/epoch010.pth',
}

for name, model in models.items():
    ckpt = checkpoints.get(name)
    if ckpt:
        model.load_state_dict(torch.load(ckpt, map_location=device))
    model.to(device).eval()
    print(f'{name} loaded')

In [ ]:
def compare(indices):
    model_names = list(models.keys())
    n_models = len(model_names)
    n_imgs = len(indices)

    # rows: 각 이미지 / cols: Original | Masked | 모델별 결과
    n_cols = 2 + n_models
    fig, axes = plt.subplots(n_imgs, n_cols, figsize=(n_cols * 3, n_imgs * 3))
    if n_imgs == 1:
        axes = [axes]

    col_titles = ['Original', 'Masked'] + model_names
    for col, title in enumerate(col_titles):
        axes[0][col].set_title(title, fontsize=12, fontweight='bold')

    with torch.no_grad():
        for row, idx in enumerate(indices):
            masked, mask, gt = dataset[idx]
            masked_d = masked.unsqueeze(0).to(device)
            mask_d   = mask.unsqueeze(0).to(device)

            axes[row][0].imshow(denorm(gt))
            axes[row][1].imshow(denorm(masked))

            for col, (name, model) in enumerate(models.items(), start=2):
                out = model(masked_d, mask_d)[0]
                axes[row][col].imshow(denorm(out))

            for ax in axes[row]:
                ax.axis('off')

    plt.tight_layout()
    plt.show()

compare([0, 1, 2, 3, 4])

In [ ]:
# 마스크 비율별 비교 (hole 비율 다른 샘플)
from datasets.mask_generator import generate_stroke_mask, get_hole_ratio

def compare_mask_ratios(img_idx=0, n_masks=5):
    _, _, gt = dataset[img_idx]
    model_names = list(models.keys())
    n_cols = 2 + len(model_names)

    fig, axes = plt.subplots(n_masks, n_cols, figsize=(n_cols * 3, n_masks * 3))
    col_titles = ['Original', 'Masked'] + model_names
    for col, title in enumerate(col_titles):
        axes[0][col].set_title(title, fontsize=12, fontweight='bold')

    with torch.no_grad():
        for row in range(n_masks):
            mask   = generate_stroke_mask(cfg.image_size, cfg.image_size)
            masked = gt * mask
            ratio  = get_hole_ratio(mask)

            masked_d = masked.unsqueeze(0).to(device)
            mask_d   = mask.unsqueeze(0).to(device)

            axes[row][0].imshow(denorm(gt))
            axes[row][0].set_ylabel(f'hole {ratio:.0%}', fontsize=10)
            axes[row][1].imshow(denorm(masked))

            for col, (name, model) in enumerate(models.items(), start=2):
                out = model(masked_d, mask_d)[0]
                axes[row][col].imshow(denorm(out))

            for ax in axes[row]:
                ax.axis('off')

    plt.tight_layout()
    plt.show()

compare_mask_ratios(img_idx=0, n_masks=5)